# 04 · Distillation: let BERTurk teach the 2.6 MB tagger

**Katip lab, notebook 4.** Smoke test 5 minutes. Quick run about 1 hour on an Apple-silicon Mac
(most of it is fine-tuning BERTurk once). Needs notebook 2.

### The problem

Katip has two ends of a size scale and nothing proven in between:

* **BERTurk**, 110.8 MB: pretrained on billions of Turkish words, so it knows what Turkish looks like.
  Too big for GitHub Pages and slow in a browser.
* **pii-names-tiny**, 2.6 MB: trained from scratch on our data only. After notebook 2 it finds about half the
  names in real text.

### The hypothesis

A gold label says only *name* or *not a name*. A teacher's probabilities say **how name-like** each
word is (*Deniz* 0.6, *önemli* 0.001). That extra signal per word is what knowledge distillation
(Hinton et al., 2015) passes on. If BERTurk's knowledge of Turkish can flow into a tiny student this
way, the student should beat pii-names-tiny **at the same 2.6 MB**, on the same training text.

### Design

| step | what | cost |
| --- | --- | --- |
| A | **Teacher BERTurk-R**: fine-tune BERTurk on exactly pii-names-tiny's data (8,000 synthetic docs + 17,967 real sentences), names only | ~35 min (MPS) |
| B | the teacher writes down its beliefs (log-probabilities) for every word of that training text | ~5 min |
| C | **students**: tiny (2.6 MB) with distillation; in full mode also *mini* (~4.6 MB) and *small* (~7.6 MB) | ~7–15 min each |
| D | score on dev, pick each model's trust setting by notebook 2's rule | ~2 min |
| C2 | **route 3**: fine-tune small models that already read Turkish: YTU Cosmos tiny BERT (~4–5 MB) and DistilBERTurk (~65–70 MB) | ~20 min |
| E | open Wiki NER test once: student vs pii-names-tiny (paired), and the size/accuracy chart | ~2 min |

Three routes to a small model that knows Turkish, all trained on the same text:

1. **from scratch** on our text (pii-names-tiny, notebook 2),
2. **distilled** from a fine-tuned BERTurk (this notebook, step C),
3. **pretrained small** models fine-tuned like BERTurk (step C2). DistilBERTurk is itself a
   distillation of BERTurk, done once on general text; Cosmos tiny has our tiny model's shape but
   was pretrained on real Turkish (lowercase only, which may cost it the capital-letter clue).

The one comparison that answers the hypothesis is **tiny + distillation vs pii-names-tiny**: same size, same
text, same recipe, the only difference is the teacher's beliefs in the loss.

In [ ]:
import json, os, pickle, random, time
from datetime import datetime, timezone
import pandas as pd
import matplotlib.pyplot as plt
import katiplab as K
import distill as D

MODE = "smoke"     # "smoke": tiny data, 1 epoch, checks every step works (5 min). Then "quick", then optionally "full".
BASE = os.environ.get("KATIPLAB_TEACHER_BASE", "dbmdz/bert-base-turkish-cased")   # the BERTurk download (~440 MB, once)
DEVICE = "auto"    # mps on a Mac, cuda on NVIDIA, else cpu

STUDENTS = {"smoke": ["tiny"], "quick": ["tiny"], "full": ["tiny", "mini", "small"]}[MODE]
ROUTE3 = {   # name: (Hugging Face id, epochs, learning rate); small models need a higher rate
    "Cosmos tiny (pretrained)": ("ytu-ce-cosmos/turkish-tiny-bert-uncased", 5, 3e-4),
    "DistilBERTurk (pretrained)": ("dbmdz/distilbert-base-turkish-cased", 2, 5e-5),
    "Cosmos mini (pretrained)": ("ytu-ce-cosmos/turkish-mini-bert-uncased", 4, 2e-4),
}
ROUTE3 = {k: v for k, v in ROUTE3.items() if MODE == "full" or "mini" not in k}
if MODE == "smoke": ROUTE3 = {k: (v[0], 1, v[2]) for k, v in ROUTE3.items() if "Cosmos tiny" in k}
if os.environ.get("KATIPLAB_ROUTE3_BASE"): ROUTE3 = {k: (os.environ["KATIPLAB_ROUTE3_BASE"], 1, v[2]) for k, v in ROUTE3.items()}
TEACHER_EPOCHS = {"smoke": 1, "quick": 2, "full": 3}[MODE]
TAG = "" if MODE != "smoke" else "_smoke"

sel = json.loads((K.RESULTS / "02_selected.json").read_text())
tiny_r_dir = K.ROOT / sel["model_dirs"][0]
print("pii-names-tiny from notebook 2:", tiny_r_dir.name, "trust", sel["trust"])

wiki = K.load_wiki("train"); random.Random(0).shuffle(wiki)      # the same shuffle and data as pii-names-tiny seed 0
synth = K.load_synth("train")
real_docs = K.pack(wiki[:sel["N"]])
if MODE == "smoke": synth, real_docs = synth[:300], real_docs[:100]
train_docs = list(synth) + list(real_docs)
print(f"mode {MODE}: {len(synth)} synthetic + {len(real_docs)} real pseudo-documents; students {STUDENTS}")

## 4.0 · Pre-registration

Written before any teacher or student exists (the smoke run writes to a separate file, so it cannot
spoil the real one).

| | hypothesis | pass if (Wiki NER test, step E) |
| --- | --- | --- |
| D1 | distillation helps at equal size | tiny student's pipeline name recall ≥ pii-names-tiny + **3 points**, McNemar p < 0.05 |
| D2 | nothing breaks | blind-set name recall ≥ **99.0%** |
| D3 | no new false alarms | ≤ **5** false names on name-free legal prose |
| D4 | still tiny | int8 ≤ **3.0 MB** |

Trust for every model is chosen on dev with notebook 2's rule (`D.choose_trust`). Route 3 has no
pass/fail line: it is reported as is, to place each option on the size chart.

In [ ]:
prereg = K.RESULTS / f"04_prereg{TAG}.json"
if not prereg.exists():
    prereg.write_text(json.dumps({"written_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"), "mode": MODE, "base": BASE,
        "D1": "tiny student pipeline recall on Wiki NER test >= pii-names-tiny + 0.03, McNemar p < 0.05",
        "D2": "blind recall >= 0.990", "D3": "negatives false names <= 5", "D4": "int8 <= 3.0 MB",
        "students": STUDENTS, "teacher_epochs": TEACHER_EPOCHS, "kd_weight": 0.5, "temperature": 2.0,
        "route3_descriptive_only": {k: v[0] for k, v in ROUTE3.items()}}, indent=1))
print(prereg.read_text())

## 4.A · Teacher: BERTurk-R

Resumable: if the folder already holds a trained teacher, this cell only loads its info. The first run
downloads BERTurk from Hugging Face (about 440 MB, cached afterwards). On an M1–M3 expect 15–20
minutes per epoch.

In [ ]:
teacher_dir = K.RUNS / f"teacher_berturk_r{TAG}"
if (teacher_dir / "train_info.json").exists():
    teacher_info = json.loads((teacher_dir / "train_info.json").read_text()); print("teacher already trained:", teacher_info)
else:
    teacher_info = D.train_teacher(teacher_dir, base=BASE, real_docs=real_docs, synth_docs=synth,
                                   epochs=TEACHER_EPOCHS, device=DEVICE)

## 4.B · The teacher's beliefs

For every word of the training text, the teacher's log-probability of *O*, *B-PER* and *I-PER*.
Cached to `runs/` so students can be retrained without re-running the teacher.

In [ ]:
cache = K.RUNS / f"soft_labels{TAG}.pkl"
if cache.exists():
    soft = pickle.loads(cache.read_bytes())
else:
    soft = D.soft_labels(teacher_dir, train_docs, device=DEVICE)
    cache.write_bytes(pickle.dumps(soft))
assert len(soft) == len(train_docs)
import numpy as np
ex = next(i for i, d in enumerate(real_docs) if any(t == "PERSON" for *_, t in d["spans"])) + len(synth)
words = [m.group() for m in D.WORD.finditer(train_docs[ex]["text"])]
print("what the teacher thinks, one real sentence (P(name)):")
print("  " + "  ".join(f"{w}:{1 - np.exp(soft[ex][k][0]):.2f}" for k, w in enumerate(words[:18])))

## 4.C · Students

Same recipe as `ml/train.py` and pii-names-tiny (OneCycle, 5 epochs, own 16k WordPiece). Loss:
**½ · cross-entropy with the gold labels + ½ · T² · KL(teacher ‖ student)** at temperature T = 2.
The temperature softens both distributions so the student also learns from the teacher's small
probabilities, which is where the "how name-like" information sits.

In [ ]:
student_dirs = {}
for arch in STUDENTS:
    out = K.RUNS / f"student_{arch}_kd{TAG}"
    if not (out / "train_info.json").exists():
        D.train_student(out, train_docs, soft=soft, arch=arch, kd_weight=0.5, temperature=2.0,
                        epochs=1 if MODE == "smoke" else 5)
    student_dirs[f"student {arch} (KD)"] = out
student_dirs

## 4.C2 · Route 3: small models that already read Turkish

Same fine-tuning code as the teacher, same training text. A model that can't be downloaded (renamed
on Hugging Face, no network) is skipped with a message, not an error.

In [ ]:
route3_dirs = {}
for name, (hf_id, epochs, lr) in ROUTE3.items():
    out = K.RUNS / ("ft_" + hf_id.split("/")[-1] + TAG)
    try:
        if not (out / "train_info.json").exists():
            D.train_teacher(out, base=hf_id, real_docs=real_docs, synth_docs=synth, epochs=epochs, lr=lr, bs=32, device=DEVICE)
        route3_dirs[name] = out
    except Exception as e:
        print(f"skipped {name} ({hf_id}): {type(e).__name__}: {str(e)[:200]}")
{k: json.loads((v / "train_info.json").read_text())["int8_mb"] for k, v in route3_dirs.items()}

## 4.D · Dev scores and trust settings

Every model, including the teacher and pii-names-tiny, gets its trust setting from the same dev rule.

In [ ]:
dev_sets = {"wiki dev": K.load_wiki("dev"), "heldout": K.load_synth("heldout")}
contenders = {"pii-names-tiny (no teacher)": tiny_r_dir, **student_dirs, **route3_dirs, "teacher BERTurk-R": teacher_dir}
chosen, dev_rows = {}, []
for name, d in contenders.items():
    rows = D.dev_table(K.Tagger(d), dev_sets)
    trust, row = D.choose_trust(rows)
    chosen[name] = trust
    dev_rows += [{"model": name, **r} for r in rows]
    print(f"{name:22} trust {trust}  dev F2 {row['wiki dev f2']:.3f}  recall {row['wiki dev recall']:.3f}  precision {row['wiki dev precision']:.3f}")
pd.DataFrame(dev_rows).to_csv(K.RESULTS / f"04_dev_scores{TAG}.csv", index=False)

## 4.E · The test, once

In [ ]:
test_sets = {"wiki test": K.load_wiki("test"), "blind": K.load_synth("blind"), "negatives": K.load_synth("negatives")}
taggers = {n: K.Tagger(d) for n, d in contenders.items()}
taggers["Tiny (published)"] = K.Tagger(K.TINY); chosen["Tiny (published)"] = None
T = {n: K.evaluate(n, K.pipeline(t, trust=chosen[n]), test_sets) for n, t in taggers.items()}
rows = []
for n, by in T.items():
    rows.append({"model": n, "MB (int8)": round(taggers[n].size_mb, 2), "trust": chosen[n],
                 "wiki test recall": f"{by['wiki test']['recall']:.1%} [{by['wiki test']['recall_ci'][0]:.1%}–{by['wiki test']['recall_ci'][1]:.1%}]",
                 "wiki test precision": f"{by['wiki test']['precision']:.1%}", "blind recall": f"{by['blind']['recall']:.1%}",
                 "false names, name-free": by["negatives"]["false"], "ms/doc": round(by["wiki test"]["ms_per_doc"], 1)})
pd.DataFrame(rows).set_index("model")

In [ ]:
A, B = T["pii-names-tiny (no teacher)"], T["student tiny (KD)"]
sig = K.paired_bootstrap(A["wiki test"]["_rows"][0], B["wiki test"]["_rows"][0])
print(f"tiny + teacher vs pii-names-tiny on real text: {100 * sig['diff']:+.1f} points (95% CI {100 * sig['ci'][0]:+.1f} to {100 * sig['ci'][1]:+.1f}), "
      f"found only by student {sig['only_b']}, only by pii-names-tiny {sig['only_a']}, McNemar p = {sig['p_mcnemar']:.2g}")
st = T["student tiny (KD)"]
verdict = {
    "D1 +3 pts vs pii-names-tiny, p<0.05": (sig["diff"] >= 0.03 and sig["p_mcnemar"] < 0.05, f"{100 * sig['diff']:+.1f} points, p={sig['p_mcnemar']:.2g}"),
    "D2 blind >= 99.0%": (st["blind"]["recall"] >= 0.99, f"{st['blind']['recall']:.1%}"),
    "D3 false names <= 5": (st["negatives"]["false"] <= 5, str(st["negatives"]["false"])),
    "D4 int8 <= 3.0 MB": (taggers["student tiny (KD)"].size_mb <= 3.0, f"{taggers['student tiny (KD)'].size_mb:.2f} MB"),
}
for h, (ok, v) in verdict.items(): print(f"{'PASS' if ok else 'FAIL'}  {h:30} {v}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for n, t in taggers.items():
    r = T[n]["wiki test"]; mb = 0.3 + t.size_mb
    ax.scatter(mb, r["recall"], s=60, color="#c2593f" if "KD" in n else "#2f7d4f" if "pretrained" in n else "#5b6b73", zorder=3)
    ax.plot([mb, mb], r["recall_ci"], color="#5b6b73", lw=1)
    ax.annotate(f"{n}\n{r['recall']:.0%}", (mb, r["recall"]), textcoords="offset points", xytext=(7, -4), fontsize=8)
ax.set_xscale("log"); ax.set_ylim(0, 1); ax.set_xlabel("download size incl. rules, MB (log)"); ax.set_ylabel("name recall, Wiki NER test")
ax.set_title("What each megabyte buys on real text"); plt.tight_layout()
plt.savefig(K.RESULTS / f"04_size_vs_recall{TAG}.png", dpi=160); plt.show()

In [ ]:
best_student = "student tiny (KD)"
(K.RESULTS / f"04_student{TAG}.json").write_text(json.dumps({"dir": str(student_dirs[best_student].relative_to(K.ROOT)), "trust": chosen[best_student]}, indent=1))
(K.RESULTS / f"04_teacher{TAG}.json").write_text(json.dumps({"dir": str(teacher_dir.relative_to(K.ROOT)), "trust": chosen["teacher BERTurk-R"]}, indent=1))
slim = {n: {s: {k: v for k, v in r.items() if k != "_rows"} for s, r in by.items()} for n, by in T.items()}
(K.RESULTS / f"04_test{TAG}.json").write_text(json.dumps({"mode": MODE, "trust": chosen, "significance": sig, "systems": slim,
    "sizes_mb": {n: t.size_mb for n, t in taggers.items()}, "verdict": {h: [ok, v] for h, (ok, v) in verdict.items()}}, indent=1, ensure_ascii=False))
print("saved; notebook 5 will pick up the teacher and the student" + (" (smoke files are ignored there)" if TAG else ""))

## What we found (fikri's Mac, 12 Nov 2026, quick mode)

This run predates step C2 (route 3), so it has routes 1 and 2 only.

| model | size (int8) | real-text recall, test (95% CI) | precision | blind recall | false names, legal prose |
| --- | --- | --- | --- | --- | --- |
| Tiny (published) | 2.57 MB | 14.8% [11.1–18.9] | 69.6% | 99.7% | 2 |
| pii-names-tiny (route 1, from scratch) | 2.57 MB | 55.5% [51.2–60.1] | 61.2% | 99.5% | 2 |
| tiny distilled from BERTurk-R (route 2) | 2.57 MB | 54.0% [49.3–58.6] | 61.8% | 99.5% | 2 |
| **BERTurk-R** (the teacher) | 110.8 MB | **90.2% [87.6–92.8]** | **90.3%** | 100% | 2 |

* **D1 fails.** Distillation did not help at 2.6 MB: −1.5 points (95% CI −4.9 to +1.7, McNemar
  p = 0.37). D2–D4 pass. The student copies the teacher on easy names and still misses the same hard
  ones, so the limit at this size is capacity and vocabulary, not supervision.
* **The surprise is the teacher.** BERTurk fine-tuned on the same mix finds **90%** of real names at
  90% precision, against 41% for the synthetic-only BERTurk pipeline. Real training text helps
  BERTurk even more than the tiny model, because BERTurk already knows Turkish and only needed to
  learn what counts as a name.
* So on real text there is a clear size/accuracy trade: **2.6 MB → 55%**, **110.8 MB → 90%**, and
  nothing measured in between yet.

## What to run next

1. **Route 3** (step C2, now in this notebook): Cosmos tiny (~4–5 MB) and DistilBERTurk (~65–70 MB,
   under GitHub's 100 MB limit) fill the gap on the chart.
2. **`MODE = "full"`**: the *mini* and *small* students show where distillation starts to pay off.
3. **Ship BERTurk-R now.** Re-run `ml/berturk/train_berturk.py` with the real sentences mixed in, so
   the Space model gets the 90% while keeping addresses and health terms.

## Reading the result

* **D1 passes**: the teacher's beliefs carry knowledge the labels don't, and a 2.6 MB model can hold
  some of it. Report "x% of BERTurk-R's real-text recall at 1/43 of the size".
* **D1 fails**: also a result. It says that at this size the bottleneck is capacity or vocabulary,
  not supervision. Then the *mini* and *small* students (full mode) show where the curve bends, and
  that is the size to ship if Pages can carry it.
* **Route 3** answers a practical question: is there a ready-made small Turkish model that beats
  both, and at what size? If Cosmos tiny (~4–5 MB) wins, the lesson is "start from pretrained, even
  when tiny". If DistilBERTurk wins clearly, it fits under GitHub's 100 MB file limit, so the demo
  could ship it instead of linking to the Space.
* Either way the chart is the selling point: how much real-text recall each megabyte buys.

After the smoke run passes, set `MODE = "quick"` in the first code cell and run all cells again.